# Monte Carlo Simulation: Sparse Design

This notebook runs Monte Carlo simulations for the Nested Cross-Validated DML-IV estimator under a **sparse** data-generating process. It defines the required data-generating function, libraries of learners, and functions for estimating the treatment effect using the PLIV score with cross-fitting and nested cross-validation.

Edit the `N_SAMPLES` and `N_SIMS` variables in the simulation code below to control the number of observations and the number of repeated simulations.

In [ ]:
import numpy as np
import pandas as pd
import os
from pathlib import Path
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import clone

# Global parameters for parallelism (adjust as needed)
CPU = os.cpu_count() or 1
MAX_WORKERS   = min(max(1, CPU - 2), 4)
TREE_N_JOBS   = 1
N_INNER_JOBS  = 1

# Cache for Toeplitz Cholesky factors
_COV_CHOL_CACHE = {}

def _toeplitz_chol(p: int, rho: float = 0.5) -> np.ndarray:
    """Compute the Cholesky factor of a Toeplitz covariance matrix with correlation rho."""
    key = (p, rho)
    if key in _COV_CHOL_CACHE:
        return _COV_CHOL_CACHE[key]
    idx = np.arange(p)
    cov = rho ** np.abs(idx[:, None] - idx[None, :])
    L = np.linalg.cholesky(cov)
    _COV_CHOL_CACHE[key] = L
    return L

def generate_data(n, p=200, design='sparse', seed=None):
    """
    Generate synthetic data for the Monte Carlo experiment under the specified design.
    
    Parameters
    ----------
    n : int
        Number of observations.
    p : int, optional
        Number of control variables W (default: 200).
    design : str, optional
        Either "sparse" or "nonlinear" to specify the data-generating process.
    seed : int, optional
        Random seed for reproducibility.
    
    Returns
    -------
    W : ndarray of shape (n, p)
        Control variables.
    Z : ndarray of shape (n, d_z)
        Instrumental variables.
    D : ndarray of shape (n,)
        Treatment variable.
    Y : ndarray of shape (n,)
        Outcome variable.
    true_pi : ndarray
        True first-stage conditional expectation of D given Z and W.
    true_q : ndarray
        True reduced-form conditional expectation of Y given Z and W.
    tau0 : float
        True treatment effect.
    """
    rng = np.random.RandomState(seed) if seed is not None else np.random.RandomState()

    L = _toeplitz_chol(p, rho=0.5)
    W = rng.normal(size=(n, p)) @ L.T

    d_z = 5  # number of instruments
    Z = rng.uniform(0.0, 1.0, size=(n, d_z))
    A = rng.normal(0.0, 1.0, size=n)

    tau0  = 1.0
    gamma = 1.0
    delta = 1.0

    if design == 'sparse':
        m_ZW = 0.5 * np.sum(Z, axis=1) + 0.5 * np.sum(W[:, :5], axis=1)
        g_W  = 0.5 * np.sum(W[:, :5], axis=1)
    elif design == 'nonlinear':
        m_ZW = Z[:, 0] * Z[:, 1] + np.sin(W[:, 0]) * np.cos(W[:, 1]) + np.exp(0.5 * W[:, 2])
        g_W  = 1 / (1 + np.exp(-W[:, 0])) + (W[:, 1] * W[:, 2]) + (W[:, 3] > 0).astype(float)
    else:
        raise ValueError("Design must be 'sparse' or 'nonlinear'")

    nu  = rng.normal(0.0, 1.0, size=n)
    eps = rng.normal(0.0, 1.0, size=n)

    D = m_ZW + gamma * A + nu
    Y = tau0 * D + g_W + delta * A + eps

    true_pi = np.full((n, d_z), 0.5)
    true_q  = m_ZW
    return W, Z, D, Y, true_pi, true_q, tau0


def get_mc_library():
    """
    Return a list of machine learning models and associated hyperparameter grids for the Monte Carlo experiments.
    Each entry in the list is a tuple (name, estimator, param_grid, needs_scaling), where
    - name is a string identifier for the model,
    - estimator is a scikit-learn regressor,
    - param_grid is a dictionary of hyperparameters for GridSearchCV (or None to skip tuning), and
    - needs_scaling is a boolean indicating whether to apply standardization.
    """
    lib = []
    lib.append(('Ridge', Ridge(), {'model__alpha': np.logspace(-2, 2, 5)}, True))
    lib.append(('Lasso', Lasso(max_iter=2000), {'model__alpha': np.logspace(-3, 0, 5)}, True))
    lib.append(('ElasticNet', ElasticNet(max_iter=2000),
                {'model__alpha': np.logspace(-3, 0, 5), 'model__l1_ratio': [0.2, 0.5, 0.8]}, True))

    lib.append(('RandomForest',
                RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=TREE_N_JOBS),
                None, False))

    lib.append(('GradientBoosting',
                GradientBoostingRegressor(n_estimators=100, random_state=42),
                None, False))

    return lib


def fit_model_and_cv_mse(X_train, y_train, name, estimator, param_grid, needs_scaling, inner_splits=3, seed=42):
    """Fit a model with (optionally) hyperparameter tuning and return the fitted model and cross-validated MSE."""
    y_train = np.asarray(y_train).reshape(-1)

    steps = []
    if needs_scaling:
        steps.append(('scaler', StandardScaler(with_mean=True, with_std=True)))
    steps.append(('model', clone(estimator)))
    pipe = Pipeline(steps)

    inner_cv = KFold(n_splits=inner_splits, shuffle=True, random_state=seed)

    if param_grid is not None:
        gs = GridSearchCV(
            pipe, param_grid, cv=inner_cv,
            scoring='neg_mean_squared_error',
            n_jobs=N_INNER_JOBS
        )
        gs.fit(X_train, y_train)
        model = gs.best_estimator_
        cv_mse = float(-gs.best_score_)
        return model, cv_mse

    # Otherwise manually compute cross-validated MSE
    mses = []
    for tr, te in inner_cv.split(X_train):
        m = clone(pipe).fit(X_train[tr], y_train[tr])
        pred = m.predict(X_train[te])
        mses.append(mean_squared_error(y_train[te], pred))
    cv_mse = float(np.mean(mses))
    model = pipe.fit(X_train, y_train)
    return model, cv_mse


def fit_predict_all_learners(X_train, y_train, X_test, library, inner_splits=3, seed=42):
    """
    Fit all learners in the library to the given training data and return predictions and CV-MSE values.
    Returns two dictionaries: predictions[name] -> predictions array, mse[name] -> cross-validated MSE.
    """
    preds = {}
    mse   = {}
    for name, estimator, param_grid, needs_scaling in library:
        model, cv_mse = fit_model_and_cv_mse(X_train, y_train, name, estimator, param_grid, needs_scaling, inner_splits, seed)
        preds[name] = model.predict(X_test)
        mse[name]   = cv_mse
    return preds, mse


def pliv_tau_se(Y, D, Z, mu_hat, r_hat, pi_hat):
    """Compute the PLIV estimator and its standard error given nuisance estimates."""
    n = len(Y)
    Y_tilde = Y - mu_hat
    D_tilde = D - r_hat
    Z_tilde = Z - pi_hat

    Omega = (Z_tilde.T @ Z_tilde) / n
    g_zd  = (Z_tilde.T @ D_tilde) / n
    g_zy  = (Z_tilde.T @ Y_tilde) / n

    Omega_inv = np.linalg.pinv(Omega)

    denom = float(g_zd.T @ Omega_inv @ g_zd)
    numer = float(g_zd.T @ Omega_inv @ g_zy)
    tau_hat = numer / denom

    u_hat = Y_tilde - tau_hat * D_tilde
    psi = Z_tilde * u_hat[:, None]
    Sigma = (psi.T @ psi) / n

    middle = float(g_zd.T @ Omega_inv @ Sigma @ Omega_inv @ g_zd)
    V = (1.0 / denom) * middle * (1.0 / denom)
    se = float(np.sqrt(V / n))
    ci = (tau_hat - 1.96*se, tau_hat + 1.96*se)
    return float(tau_hat), float(se), ci


def run_one_sim_all_models(W, Z, D, Y, tau0, library, k_outer=3, inner_splits=3, seed=42):
    """
    Run one simulation across all models: perform cross-fitting and compute PLIV estimates for each learner
    as well as the adaptive nested-CV learner.
    """
    n, d_z = Z.shape
    kf = KFold(n_splits=k_outer, shuffle=True, random_state=seed)

    learner_names = [m[0] for m in library]

    mu_by = {name: np.zeros(n) for name in learner_names}
    r_by  = {name: np.zeros(n) for name in learner_names}
    pi_by = {name: np.zeros((n, d_z)) for name in learner_names}

    mu_nested = np.zeros(n)
    r_nested  = np.zeros(n)
    pi_nested = np.zeros((n, d_z))

    chosen_nested = {"mu": [], "r": [], "pi": [[] for _ in range(d_z)]}

    for fold, (tr, te) in enumerate(kf.split(W)):
        W_tr, W_te = W[tr], W[te]
        Y_tr, D_tr, Z_tr = Y[tr], D[tr], Z[tr]

        preds_mu, mse_mu = fit_predict_all_learners(W_tr, Y_tr, W_te, library, inner_splits, seed=1000 + fold)
        for name in learner_names:
            mu_by[name][te] = preds_mu[name]
        best_mu = min(mse_mu, key=mse_mu.get)
        mu_nested[te] = preds_mu[best_mu]
        chosen_nested["mu"].append(best_mu)

        preds_r, mse_r = fit_predict_all_learners(W_tr, D_tr, W_te, library, inner_splits, seed=2000 + fold)
        for name in learner_names:
            r_by[name][te] = preds_r[name]
        best_r = min(mse_r, key=mse_r.get)
        r_nested[te] = preds_r[best_r]
        chosen_nested["r"].append(best_r)

        for j in range(d_z):
            preds_pj, mse_pj = fit_predict_all_learners(W_tr, Z_tr[:, j], W_te, library, inner_splits, seed=3000 + 37*j + fold)
            for name in learner_names:
                pi_by[name][te, j] = preds_pj[name]
            best_pj = min(mse_pj, key=mse_pj.get)
            pi_nested[te, j] = preds_pj[best_pj]
            chosen_nested["pi"][j].append(best_pj)

    results = {}
    for name in learner_names:
        tau_hat, se_hat, _ = pliv_tau_se(Y, D, Z, mu_by[name], r_by[name], pi_by[name])
        ci_low = tau_hat - 1.96*se_hat
        ci_up  = tau_hat + 1.96*se_hat
        results[name] = {"tau_hat": tau_hat, "SE": se_hat, "Coverage": int(ci_low <= tau0 <= ci_up)}

    tau_hat, se_hat, _ = pliv_tau_se(Y, D, Z, mu_nested, r_nested, pi_nested)
    ci_low = tau_hat - 1.96*se_hat
    ci_up  = tau_hat + 1.96*se_hat
    results['Adaptive Nested-CV'] = {
        'tau_hat': tau_hat, 'SE': se_hat, 'Coverage': int(ci_low <= tau0 <= ci_up),
        'Chosen(mu)': chosen_nested['mu'], 'Chosen(r)': chosen_nested['r'],
    }
    return results


In [ ]:
# Simulation parameters
DESIGN = "sparse"
N_SAMPLES = [1000, 3000, 10000]  # sample sizes
N_SIMS   = 10  # number of Monte Carlo repetitions (increase for better precision)

results = []
library = get_mc_library()

for n in N_SAMPLES:
    for sim in range(N_SIMS):
        W, Z, D, Y, true_pi, true_q, tau0 = generate_data(n, design=DESIGN, seed=sim)
        res = run_one_sim_all_models(W, Z, D, Y, tau0, library)
        for model, r in res.items():
            results.append({
                'Design': DESIGN,
                'N': n,
                'Model': model,
                'tau_hat': r['tau_hat'],
                'SE': r['SE'],
                'Coverage': r['Coverage']
            })

# Convert results into a DataFrame for summary
import pandas as pd
df_results = pd.DataFrame(results)

# Summarize mean estimate, bias, standard deviation, average SE, and coverage
tau0 = 1.0
summary = []
for model in df_results['Model'].unique():
    for n in N_SAMPLES:
        sub = df_results[(df_results['Model'] == model) & (df_results['N'] == n)]
        tau_vals = sub['tau_hat'].astype(float).to_numpy()
        se_vals  = sub['SE'].astype(float).to_numpy()
        cov_vals = sub['Coverage'].astype(float).to_numpy()
        summary.append({
            'Model': model,
            'N': n,
            'Mean(tau_hat)': np.mean(tau_vals),
            'Bias(tau_hat)': np.mean(tau_vals - tau0),
            'SD(tau_hat)': np.std(tau_vals),
            'Avg(SE)': np.mean(se_vals),
            'Coverage(%)': 100 * np.mean(cov_vals)
        })

df_summary = pd.DataFrame(summary)
print(df_summary)

# Optional: pivot tables for easier interpretation
summary_table = df_summary.pivot(index='Model', columns='N', values=['Mean(tau_hat)', 'Bias(tau_hat)', 'SD(tau_hat)', 'Avg(SE)', 'Coverage(%)'])
summary_table
